# Конспект. Модуль 8: Метрики — как измерять качество рекомендаций

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 8 из 13 — «Метрики: как измерять качество рекомендаций»
**Цель модуля:** получить полный, формально обоснованный инструментарий для количественной оценки качества рекомендательной системы. До сих пор в курсе мы оценивали результаты интуитивно («предсказание выглядит разумным», «шесть методов согласились», Модуль 6.4.3) или простым RMSE (Модуль 5). Этот модуль формализует метрики **ранжирования** — то, что реально нужно для честного сравнения моделей и обоснованного подбора гиперпараметров вроде `α` в Weighted Hybrid (Модуль 7.3.1).

**Связь с предыдущими модулями:** все метрики этого раздела применяются к **упорядоченному списку** рекомендаций — тому самому «топ-K», который в разных формах уже возвращали `ContentBasedRecommender.recommend()` (Модуль 6.5.1), функция `get_candidates()` (прообраз из Модуля 5.5.5) и все остальные реализованные вами функции рекомендаций.

## 8.1 Почему обычные метрики не подходят

### 8.1.1 Accuracy бессмысленна — напоминание и углубление

В Модуле 1.5.3 (Popularity Bias) мы уже увидели, что метрика точности может расти при деградации реальной полезности системы. Здесь та же проблема формулируется точнее: задача RecSys — не бинарная классификация одного объекта («кликнет / не кликнет»), а **ранжирование списка**. Accuracy как метрика классификации в принципе не имеет понятия «порядка» — она не может ответить на вопрос «а был ли самый релевантный товар показан первым, а не пятым?».

### 8.1.2 Ключевое новое требование — учёт порядка

Ни одна из метрик классификации, с которыми вы уже работали (Precision, Recall, F1, ROC-AUC — Неделя 4 общего плана), не различает ситуацию «релевантный товар на позиции 1» от «релевантный товар на позиции 10» — обе воспринимаются одинаково: «был найден». Но для пользователя, который обычно смотрит первые несколько позиций выдачи, это принципиально разные результаты. Весь этот модуль, по сути, — набор способов **адаптировать** уже знакомые вам метрики (Precision, Recall) под понятие top-K списка (8.2–8.3), и затем **пойти дальше**, введя метрики, чувствительные к порядку (8.4–8.6).

## 8.2 Precision@K и Recall@K

### 8.2.1 Формулы

In [ ]:
Precision@K = (число релевантных товаров в top-K) / K
Recall@K    = (число релевантных товаров в top-K) / (общее число релевантных товаров у пользователя)

**Отличие от классического Precision/Recall (которое вы уже знаете):** classической формуле безразлично, сколько объектов вообще было предсказано положительными — она считается по всем предсказанным как «позитив». Здесь же знаменатель `Precision@K` — это **фиксированное** число `K` (размер выдачи), независимо от того, сколько из них реально релевантны. Это отражает реальное ограничение интерфейса — сайт показывает ровно 10 позиций, не больше и не меньше.

### 8.2.2 Полный проверенный численный пример

Пусть для целевого пользователя `U` истинно релевантные товары (то есть те, с которыми он реально взаимодействовал в отложенной test-выборке — прямая связь с Модулем 1.7, temporal split) — это множество `{A, C, F, H, J}` — всего **5** релевантных товаров.

Модель выдала следующий ранжированный топ-10:

In [ ]:
Позиция:     1    2    3    4    5    6    7    8    9    10
Товар:       B    A    C    D    E    F    G    H    I    K
Релевантен:  0    1    1    0    0    1    0    1    0    0

**Расчёт для разных значений K (реально вычислено):**

| K | Precision@K | Recall@K | Комментарий |
|:---:|:---:|:---:|:---|
| 1 | 0.000 | 0.000 | Первая позиция (B) — нерелевантна |
| 3 | 0.667 | 0.400 | 2 из 3 первых — релевантны (A, C) |
| 5 | 0.400 | 0.400 | Всё ещё те же 2 релевантных, знаменатель Precision вырос |
| 10 | 0.400 | 0.800 | Найдено 4 из 5 релевантных (J так и не появился в топ-10) |

### 8.2.3 Компромисс между Precision и Recall при росте K

Обратите внимание на характерную динамику из таблицы: при `K=3->5` Precision **упало** (0.667->0.400), потому что мы добавили две нерелевантные позиции (D, E), не найдя новых релевантных; при `K=5->10` Precision осталось на месте (0.400), а Recall выросло вдвое (0.400->0.800), потому что среди новых пяти позиций нашлись 2 релевантных (F, H) ровно в той же пропорции. Это иллюстрирует общее правило: **при увеличении K Recall монотонно не убывает** (мы только добавляем позиции, релевантные товары не могут «исчезнуть» из списка), тогда как **Precision может как расти, так и падать** — в зависимости от того, насколько «удачные» именно новые позиции добавляются.

### 8.2.4 Товар J — важное наблюдение

Обратите внимание: товар `J` — релевантный (входит в ground truth), но **ни разу не появился** в топ-10 модели. Это напрямую отражается в итоговом `Recall@10=0.800` (а не `1.000`) — модель **пропустила** одну из пяти вещей, которые реально были бы интересны пользователю. Ни `Precision@K`, ни любая метрика, считающая только по видимому списку, не увидела бы эту проблему без явного использования полного ground truth — что подчёркивает важность корректно организованной test-выборки (Модуль 1.7).

## 8.3 F1@K

### 8.3.1 Формула

Гармоническое среднее — та же самая конструкция F1, что вы уже знаете из классификации:

In [ ]:
F1@K = 2 × Precision@K × Recall@K / (Precision@K + Recall@K)

### 8.3.2 Расчёт на том же примере

| K | Precision@K | Recall@K | F1@K |
|:---:|:---:|:---:|:---:|
| 3 | 0.667 | 0.400 | 0.500 |
| 5 | 0.400 | 0.400 | 0.400 |
| 10 | 0.400 | 0.800 | 0.533 |

**Наблюдение:** `F1@10` (0.533) выше, чем `F1@5` (0.400), несмотря на одинаковый `Precision`, — потому что гармоническое среднее «награждает» за существенный прирост Recall при неизменной Precision. Как и в классической постановке, F1 полезен, когда нужен единственный компромиссный показатель, но скрывает, за счёт чего именно достигнут баланс — на практике Precision@K и Recall@K чаще анализируют раздельно, а F1@K приводят как сводную цифру для отчётности.

## 8.4 MAP (Mean Average Precision)

### 8.4.1 Зачем нужна метрика «умнее», чем Precision@K

`Precision@K` не видит **порядок** релевантных товаров внутри топ-K — список `[relevant, relevant, junk, junk]` и список `[junk, junk, relevant, relevant]` дадут одинаковый `Precision@4`, хотя первый явно лучше (релевантное — выше). MAP решает эту проблему.

### 8.4.2 Формула Average Precision (AP) для одного пользователя

In [ ]:
AP = (1/R) × Σ_{k: rel(k)=1} Precision@k

где `R` — **общее** число релевантных товаров у пользователя (а не только найденных в списке!), суммирование идёт только по тем позициям `k`, где оказался релевантный товар, и на каждой такой позиции считается `Precision@k` (используя ровно `k`, а не полную длину списка).

**Критически важная деталь, часто упускаемая:** делим именно на `R` (все релевантные товары пользователя), а не на число **найденных** релевантных товаров в списке. Это гарантирует, что пропущенные релевантные товары (как `J` в нашем примере) **штрафуют** итоговую метрику, а не просто исключаются из расчёта.

### 8.4.3 Полный проверенный численный пример (2 пользователя)

**Пользователь 1** (наш пример из 8.2.2), релевантные позиции — ранги 2 (A), 3 (C), 6 (F), 8 (H):

In [ ]:
Precision@2 = 1/2 = 0.500
Precision@3 = 2/3 = 0.667
Precision@6 = 3/6 = 0.500
Precision@8 = 4/8 = 0.500

AP_user1 = (0.500 + 0.667 + 0.500 + 0.500) / 5 = 2.167 / 5 = 0.4333

(делим на `R=5`, включая пропущенный товар `J` — отсюда `AP < 1`, даже несмотря на то, что все 4 найденных релевантных товара были ранжированы достаточно рано)

**Пользователь 2**, релевантные товары `{P, Q, R}` (`R=3`), выдача `[P, S, Q, T, U]` — релевантны позиции 1 (P) и 3 (Q), товар `R` не найден:

In [ ]:
Precision@1 = 1/1 = 1.000
Precision@3 = 2/3 = 0.667

AP_user2 = (1.000 + 0.667) / 3 = 1.667 / 3 = 0.5556

**Итоговый MAP:**

In [ ]:
MAP = (AP_user1 + AP_user2) / 2 = (0.4333 + 0.5556) / 2 = 0.4944

### 8.4.4 Почему MAP «умнее» Precision@K

Сравните пользователя 2 с гипотетической альтернативной выдачей `[S, T, P, U, Q]` (те же самые релевантные товары `P` и `Q`, но на менее ранних позициях — 3 и 5 вместо 1 и 3): `Precision@5` для обоих вариантов был бы одинаковым (`2/5 = 0.4`), но `AP` для второго варианта оказался бы **ниже**, потому что релевантные товары оказались на менее ранних позициях. Именно эта чувствительность к позиции внутри списка и есть главное преимущество MAP над Precision@K.

## 8.5 NDCG (Normalized Discounted Cumulative Gain) — золотой стандарт

### 8.5.1 Зачем ещё одна метрика, если уже есть MAP

MAP расширяется естественным образом только на **бинарную** релевантность (товар либо релевантен, либо нет). Но на практике релевантность часто градуальна — рейтинг может быть 1–5, а не 0/1, и «хочется» получить кредит пропорционально силе релевантности, а не бинарно. NDCG изначально спроектирован для работы с **градуальной** релевантностью (хотя прекрасно работает и с бинарной, как в нашем примере).

### 8.5.2 Формулы — CG, DCG, IDCG, NDCG

In [ ]:
CG@K = Σ_{i=1}^{K} rel_i                              (без учёта позиции)
DCG@K = Σ_{i=1}^{K} rel_i / log2(i + 1)                (с логарифмическим штрафом за позицию)
IDCG@K = DCG@K для идеального (правильно отсортированного) списка
NDCG@K = DCG@K / IDCG@K                                (нормировка в диапазон [0, 1])

**Интуиция логарифмического знаменателя:** на позиции 1 (`i=1`) знаменатель `log2(2)=1` — почти без штрафа. На позиции 10 (`i=10`) знаменатель `log2(11)≈3.46` — товар «весит» втрое меньше. Это математически формализует то же самое интуитивное наблюдение, которое мы уже сделали в 8.4.4 на MAP, но делает это плавно и явно для каждой позиции, а не только косвенно через накопленный Precision.

### 8.5.3 Полный проверенный численный пример (продолжение примера 8.2.2)

Используем relevance-вектор `[0,1,1,0,0,1,0,1,0,0]` (пользователь 1).

**DCG@10 (реально вычислено):**

In [ ]:
позиция 2: 1/log2(3) = 1/1.585 = 0.6309
позиция 3: 1/log2(4) = 1/2.000 = 0.5000
позиция 6: 1/log2(7) = 1/2.807 = 0.3562
позиция 8: 1/log2(9) = 1/3.170 = 0.3155

DCG@10 = 0.6309 + 0.5000 + 0.3562 + 0.3155 = 1.8026

**IDCG@10** — идеальный список: все `min(R,K)=min(5,10)=5` релевантных товаров стоят на позициях 1–5 подряд:

In [ ]:
IDCG@10 = 1/log2(2) + 1/log2(3) + 1/log2(4) + 1/log2(5) + 1/log2(6)
        = 1.0000 + 0.6309 + 0.5000 + 0.4307 + 0.3869
        = 2.9485

**Итоговый NDCG@10:**

In [ ]:
NDCG@10 = 1.8026 / 2.9485 = 0.6114

### 8.5.4 Интерпретация и почему это «золотой стандарт»

`NDCG@10 = 0.611` означает: наш список набрал **61.1%** от максимально возможного «качества ранжирования» при этом наборе релевантных товаров. Сравните с `Precision@10=0.400` — NDCG даёт более информативную, чувствительную к позиции оценку **того же самого списка**. Именно эта чувствительность к позиции, нормировка в фиксированный диапазон `[0,1]` (удобно для сравнения между пользователями с разным числом релевантных товаров — в отличие от сырого `DCG`, который зависит от `R`) и естественная поддержка градуальной релевантности сделали NDCG стандартом де-факто в индустрии поисковых и рекомендательных систем — вы встретите её как основную целевую метрику практически в любой production-задаче ранжирования (включая LightGBM `objective='lambdarank'` в Модуле 9.5, где NDCG оптимизируется напрямую).

## 8.6 MRR (Mean Reciprocal Rank)

### 8.6.1 Формула и область применения

In [ ]:
RR = 1 / rank_первого_релевантного_элемента
MRR = среднее RR по всем пользователям

MRR — метрика для задач другого типа: не «дай хороший список из K релевантных», а **«найди хотя бы один правильный ответ, и чем раньше — тем лучше»**. Типичный пример — не столько рекомендации в привычном смысле, сколько поиск (Модуль 1.1.2): если пользователь ищет конкретный товар, важен только момент, когда он впервые увидел нужный результат, а не полное качество всего списка.

### 8.6.2 Полный проверенный численный пример

In [ ]:
Пользователь 1: первый релевантный товар — на позиции 2 (A) -> RR_1 = 1/2 = 0.500
Пользователь 2: первый релевантный товар — на позиции 1 (P) -> RR_2 = 1/1 = 1.000

MRR = (0.500 + 1.000) / 2 = 0.750

**Когда MRR предпочтительнее NDCG/MAP:** если бизнес-задача действительно устроена как «нужен только один хороший ответ» (например, автодополнение поискового запроса, ответ на вопрос) — MRR прямо измеряет то, что важно. Если задача — «нужен качественный список из нескольких позиций» (типичная лента рекомендаций) — MAP/NDCG информативнее, потому что MRR **полностью игнорирует** всё, что происходит после первого найденного релевантного товара (сравните: `RR_1=0.5` не изменился бы, даже если бы после позиции 2 не было найдено вообще ни одного релевантного товара — а ведь у нас их ещё 3).

## 8.7 Дополнительные метрики качества списка

Эти метрики оценивают не точность отдельно взятой рекомендации, а системные, «качественные» свойства — прямое развитие тем, поднятых в Модуле 1.5.5 (Novelty, Diversity, Serendipity) и Модуле 1.5.3 (Popularity Bias).

### 8.7.1 Coverage (покрытие каталога)

In [ ]:
Coverage = (число уникальных товаров, когда-либо порекомендованных хотя бы одному пользователю) / (общий размер каталога)

**Пример:** каталог из 1000 товаров; проанализировав рекомендации, выданные всем пользователям системы за период, обнаружили, что среди всех топ-10 суммарно встретилось только 150 уникальных товаров -> `Coverage = 150/1000 = 15%`. Это прямой количественный индикатор Popularity Bias (Модуль 1.5.3, 12.2) — низкий Coverage означает, что система эксплуатирует лишь малую часть каталога, независимо от того, насколько высок Precision/NDCG для отдельных пользователей.

### 8.7.2 Diversity (внутрисписочное разнообразие) — с использованием векторов из Модуля 6

In [ ]:
Diversity(список) = среднее попарное (1 - cosine_similarity) между всеми парами товаров в списке

**Полный проверенный численный пример** — переиспользуем TF-IDF векторы товаров из Модуля 6.3.3:

**Список A: `[I1, I4, I3]`** (напомним, I1 и I4 — идентичны по жанрам, Модуль 6.3.3):

In [ ]:
dissim(I1,I4) = 1 - 1.0000 = 0.0000   (идентичные товары!)
dissim(I1,I3) = 1 - 0.2371 = 0.7629
dissim(I4,I3) = 1 - 0.2371 = 0.7629

Diversity = (0.0000 + 0.7629 + 0.7629) / 3 = 0.5086

**Список B: `[I1, I2, I5]`**:

In [ ]:
dissim(I1,I2) = 1 - 0.0000 = 1.0000   (полностью разные жанры)
dissim(I1,I5) = 1 - 0.0000 = 1.0000
dissim(I2,I5) = 1 - 0.3498 = 0.6502

Diversity = (1.0000 + 1.0000 + 0.6502) / 3 = 0.8834

**Интерпретация:** список A (Diversity=0.509) содержит фактически дублирующий контент (I1≈I4) — низкое разнообразие, даже если оба товара по отдельности отлично подходят пользователю (высокая релевантность не спасает от низкого разнообразия!). Список B (Diversity=0.883) охватывает три разных жанровых направления. **Важный практический вывод:** Precision/NDCG никак не отличили бы эти два списка, если бы, скажем, I1, I4 и I3 (список A) все были одинаково релевантны пользователю по отдельности — только Diversity явно показывает, что список A практически «показывает одно и то же дважды».

### 8.7.3 Novelty (новизна)

Простой и широко используемый вариант — через **self-information** (та же математическая идея, что лежит в основе энтропии и IDF из Модуля 6.3.2 — редкое несёт больше информации):

In [ ]:
Novelty(i) = -log2(popularity_i)

где `popularity_i` — доля пользователей, взаимодействовавших с товаром `i`.

**Проверенный пример:**

In [ ]:
Novelty(I1) при популярности 80% (популярный хит) = -log2(0.80) = 0.3219
Novelty(I5) при популярности 5% (нишевый товар)    = -log2(0.05) = 4.3219

I5 почти в 13 раз «новее» I1 по этой мере — прямое количественное выражение интуиции из Модуля 1.5.5.

### 8.7.4 Serendipity (приятная неожиданность)

In [ ]:
Serendipity(u, i) = relevance(u, i) × unexpectedness(u, i)

где `unexpectedness` обычно определяется как несходство с «ожидаемой», предсказуемой рекомендацией (например, с тем, что предложила бы простая popularity-based или чисто контентная модель — Модуль 6). **Пример логики:** товар, который пользователь оценил высоко (`relevance` высокий), но который **не** входил бы в предсказуемую топ-выдачу простого baseline (`unexpectedness` высокий), получает высокий Serendipity. Товар, который пользователь оценил высоко, но который и baseline легко бы порекомендовал (например, самый популярный товар в его любимой категории), получает низкий Serendipity, несмотря на высокую релевантность — именно произведение (а не сумма!) двух факторов гарантирует, что **оба** условия должны выполняться одновременно.

## 8.8 Практика

### 8.8.1 Полная реализация всех метрик с нуля

In [ ]:
import numpy as np

def precision_at_k(relevance: list, k: int) -> float:
    return sum(relevance[:k]) / k

def recall_at_k(relevance: list, k: int, total_relevant: int) -> float:
    return sum(relevance[:k]) / total_relevant

def f1_at_k(relevance: list, k: int, total_relevant: int) -> float:
    p = precision_at_k(relevance, k)
    r = recall_at_k(relevance, k, total_relevant)
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0

def average_precision(relevance: list, total_relevant: int) -> float:
    hits, precisions = 0, []
    for k, rel in enumerate(relevance, start=1):
        if rel == 1:
            hits += 1
            precisions.append(hits/k)
    return sum(precisions)/total_relevant if precisions else 0.0

def dcg_at_k(relevance: list, k: int) -> float:
    relevance = relevance[:k]
    return sum(rel/np.log2(i+2) for i, rel in enumerate(relevance))

def ndcg_at_k(relevance: list, k: int, total_relevant: int) -> float:
    dcg = dcg_at_k(relevance, k)
    ideal_hits = min(total_relevant, k)
    idcg = sum(1/np.log2(i+2) for i in range(ideal_hits))
    return dcg/idcg if idcg > 0 else 0.0

def reciprocal_rank(relevance: list) -> float:
    for i, rel in enumerate(relevance, start=1):
        if rel == 1:
            return 1/i
    return 0.0

def diversity(item_ids: list, item_vectors: dict) -> float:
    from itertools import combinations
    dissims = []
    for a, b in combinations(item_ids, 2):
        va, vb = item_vectors[a], item_vectors[b]
        cos = np.dot(va, vb) / (np.linalg.norm(va)*np.linalg.norm(vb) + 1e-10)
        dissims.append(1 - cos)
    return float(np.mean(dissims)) if dissims else 0.0

### 8.8.2 Сравнение 3 моделей курса на общей метрике

- Взять реализации UB-CF (Модуль 3.4.1), IB-CF (Модуль 4.5.1) и explicit ALS (Модуль 5.7.1) — обучить на MovieLens с temporal split (Модуль 1.7).
- Для каждого пользователя в test-части построить top-10 рекомендаций каждой моделью и посчитать Precision@10, Recall@10, MAP, NDCG@10.
- Свести результаты в единую таблицу — обычно на реальных данных матричная факторизация (Модуль 5) даёт заметно более высокий NDCG, чем memory-based методы (Модули 3–4), особенно на пользователях с небольшим числом оценок — прямое практическое подтверждение теоретических аргументов о разреженности (Модули 1.4, 3.3.1).
- **Дополнительно:** посчитать Coverage (8.7.1) для каждой модели по всем пользователям test-выборки — часто модель с лучшим NDCG имеет **худший** Coverage (более уверенно рекомендует уже популярное) — обсудить этот компромисс явно, письменно, как подготовку к вопросу на собеседовании.

### 8.8.3 Вопросы для самопроверки

1. В разделе 8.4.2 подчёркивается, что AP делится на `R` (все релевантные товары пользователя), а не на число найденных. Придумайте пример, где два списка дают одинаковый `Precision@K`, но разный AP из-за этой детали.
2. Объясните, почему `IDCG@10` в нашем примере (8.5.3) использует `min(R,K)=5`, а не просто `K=10` релевантных позиций в идеальном списке — что физически невозможно, если предположить `K` релевантных позиций при `R=5`?
3. Приведите пример (возможно, гипотетический) ситуации, где модель с высоким NDCG@10 может иметь **низкий** Coverage (8.7.1) по всей системе — как это связано с материалом Модуля 1.5.3?
4. В разделе 8.7.2 мы показали, что список `[I1, I4, I3]` имеет низкую Diversity, потому что I1 и I4 — идентичные по жанрам товары. Если бы Item-Based CF (Модуль 4) использовался для генерации топ-3 рекомендаций на основе высокой оценки I1, почему именно этот алгоритм — по своей природе — рискует выдать список с низким Diversity чаще, чем, скажем, гибридная модель из Модуля 7?

## Глоссарий модуля 8

| Термин | Короткое определение |
|:---|:---|
| Precision@K | Доля релевантных товаров среди первых K рекомендованных |
| Recall@K | Доля всех релевантных товаров пользователя, найденных в top-K |
| F1@K | Гармоническое среднее Precision@K и Recall@K |
| AP (Average Precision) | Precision, усреднённый по позициям релевантных товаров, с делением на все релевантные |
| MAP | Среднее AP по всем пользователям |
| DCG | Сумма релевантностей с логарифмическим штрафом за позицию |
| IDCG | DCG идеального (правильно отсортированного) списка |
| NDCG | DCG / IDCG — нормированная метрика ранжирования, чувствительная к позиции |
| MRR | Среднее по пользователям от `1/позиция первого релевантного товара` |
| Coverage | Доля каталога, которую система вообще способна порекомендовать |
| Diversity | Среднее попарное несходство товаров внутри одного списка |
| Novelty | Мера непопулярности рекомендованного товара (self-information) |
| Serendipity | Произведение релевантности и неожиданности рекомендации |

**Связь со следующим модулем:** до сих пор метрики (этот модуль) использовались только для **оценки** уже построенных моделей — после того, как рекомендации сгенерированы каким-то из алгоритмов Модулей 3–7. Модуль 9 делает следующий логичный шаг: что, если **обучать модель напрямую оптимизировать** одну из этих метрик (чаще всего — NDCG) вместо косвенных прокси вроде RMSE? Это и есть идея Learning to Rank.